# Figure 2 - ABCDEF



In [ ]:
from libraries import *
from parameters import *

In [ ]:
%load_ext rpy2.ipython

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
adata = sc.read("outputs/anndata/adata-hash-features_singlets_05242020.h5ad")

In [ ]:
adata.obs["subcellType"] = "DC2"
adata.obs.loc[adata.obs["leiden"] == "8","subcellType"] = "DC1"
adata.obs.loc[adata.obs["leiden"] == "5","subcellType"] = "mReg"
adata.obs.loc[adata.obs["leiden"] == "3","subcellType"] = "Mac"

In [ ]:
k = adata.obs["subcellType"].value_counts()

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='leiden', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', 
           ax=ax, show=False, size=0.3);

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='leiden',  save="Figure_1C.pdf", legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=0.3);

In [ ]:
sc.tl.dendrogram(adata, groupby='leiden')

In [ ]:
sc.pl.dendrogram(adata, groupby='leiden', save="SupFig_1F.pdf")

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", n_genes=2000, method="t-test_overestim_var", use_raw =False)

In [ ]:
markerGenes = pd.DataFrame(adata.uns['rank_genes_groups']['names'])

In [ ]:
markerGenes = markerGenes.iloc[0:8,:]

In [ ]:
markerGenes = markerGenes.values.flatten(order='F')

In [ ]:
sc.pl.dotplot(adata, markerGenes, groupby='leiden', cmap='bwr', figsize=(16,4), 
              vmin=-3, vmax=3, use_raw=False,  dot_min=0.1, dot_max=1, save="SupFig_1G.pdf")

In [ ]:
#markerGenes = np.unique(markerGenes.values.flatten())

In [ ]:
# sc.pl.matrixplot(adata, var_names = markerGenes, groupby='leiden', dendrogram=True,
#                       use_raw=False, vmin=-3, vmax=3,cmap='bwr',  swap_axes=False, figsize=(16,4))

In [ ]:
# sc.pl.rank_genes_groups_matrixplot(adata, n_genes=10, standard_scale='var', cmap='Blues')

In [ ]:
gene_list_url = 'https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt'

cell_cycle_genes = [str(x.strip(), 'utf-8').capitalize() for x in urlopen(gene_list_url)] # capitalize = shame


s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]


sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
sc.pl.umap(adata, color='phase', palette = "Paired", save="Figure_1G.pdf" )

In [ ]:
f, ax = plt.subplots(2, 2, figsize=(20, 8))
sc.pl.violin(adata, keys='log10_n_umis', groupby='leiden', rotation=90,   ax=ax[0][0],show=False, stripplot=False)
sc.pl.violin(adata, keys='mt_frac', groupby='leiden', rotation=90,   ax=ax[0][1],show=False, stripplot=False)
sc.pl.violin(adata, keys='n_genes', groupby='leiden', rotation=90,   ax=ax[1][0],show=False, stripplot=False)


In [ ]:
dcGenes = pd.read_csv('./PositiveControls/DC_cellstate_genes.csv')

In [ ]:
dc1Genes = dcGenes["DC1 genes"].unique()
sc.tl.score_genes(adata=adata, gene_list=dc1Genes, score_name="DC1")

In [ ]:
sc.pl.umap(adata, color="DC1", size=1, color_map="coolwarm", save="Figure_1F.pdf", vmax=0.15, vmin=-0.15)


In [ ]:
#sc.pl.violin(adata, "DC1", groupby='leiden')

In [ ]:
dc2Genes = dcGenes["DC2 genes"].unique()
sc.tl.score_genes(adata=adata, gene_list=dc2Genes, score_name="DC2")

In [ ]:
sc.pl.umap(adata, color="DC2", size=1, color_map="coolwarm", save="Figure_1D.pdf")


In [ ]:
mregGenes = dcGenes["mregDC genes"].unique()
sc.tl.score_genes(adata=adata, gene_list=mregGenes, score_name="mreg")


In [ ]:
sc.pl.umap(adata, color="mreg", size=1, color_map="coolwarm", save="Figure_1E.pdf")


In [ ]:
macGenes = dcGenes["Macrophage genes"].unique()
sc.tl.score_genes(adata=adata, gene_list=macGenes, score_name="Mac")


In [ ]:
sc.pl.umap(adata, color="Mac", size=1, color_map="coolwarm", save="SupFig_1Y.pdf")


In [ ]:
allDCgenes = np.concatenate((dc1Genes, dc2Genes, mregGenes))
sc.tl.score_genes(adata=adata, gene_list=allDCgenes, score_name="DCSig")

In [ ]:
sc.pl.umap(adata, color="DCSig", size=1, color_map="coolwarm", 
           vmin=-0.2, vmax=0.3, save="SupFig_1X.pdf")

In [ ]:
adata.obs["DCSig_zscore"] = scipy.stats.zscore(adata.obs["DCSig"])
adata.obs["Mac_zscore"] = scipy.stats.zscore(adata.obs["Mac"])

In [ ]:
adata.obs["MACoverDC"] = adata.obs["Mac_zscore"] - adata.obs["DCSig_zscore"]

In [ ]:
sc.pl.umap(adata, color="MACoverDC", size=1, color_map="PiYG", vmin=-5, vmax=5, save="Figure_2F.pdf")
